<h1>Chapter 11 - Coding Agents</h1>
<i>Create an Agent for developing code.</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 11 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [1]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [2]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [3]:
import os
from illustrated_agents.chapters.ch5_native import LLM

# Ollama through OpenAI API
llm = LLM(model="gemma4:e4b", backend="openai", api_base="http://localhost:11434/v1/", think=True)

# Llama.cpp server
# llm = LLM(model="openai/gemma-4-E4B-it-Q4_K_M", backend="litellm", api_base="http://localhost:8080", think=True)

# LM Studio
# llm = LLM(model="lm_studio/gemma-4-E4B-it", backend="litellm", api_base="http://localhost:1234/v1", think=True)

# Google's Gemini / Gemma
# os.environ['GEMINI_API_KEY'] = "YOUR_API_KEY"
# llm = LLM(model="gemini/gemini-2.5-flash", backend="litellm", api_key=None)

## 2 - Coding Tools


As we covered in this chapter, code tools are what tie an LLM to a software system. The two main categories are :

* **file manipulation tools**: This lets the agent read, list, and write files
* **code interpreter**: This lets the agent execute code it writes. 

Below, we define four tools that cover both categories: `read_file` and `list_files` allow the agent to explore existing files, `write_file` lets it create or modify files, and `execute_python` runs Python code in a subprocess with a timeout. Finally, we also add the `show_python` function that allows the agent to show formatted code.



In [4]:
import subprocess
import sys
from pathlib import Path

# For the `show_python` function
from pygments import highlight
from pygments.lexers import PythonLexer
from pygments.formatters import TerminalFormatter

def read_file(path: str) -> str:
    """Read a file's contents."""
    target = Path(path)
    if not target.exists():
        return f"Error: '{path}' not found."
    return target.read_text(encoding="utf-8")


def list_files(directory: str = ".") -> str:
    """List files in a directory."""
    target = Path(directory)
    if not target.is_dir():
        return f"Error: '{directory}' is not a directory."
    entries = sorted(target.iterdir())
    return "\n".join(p.name + ("/" if p.is_dir() else "") for p in entries) or "(empty)"


def write_file(path: str, content: str) -> str:
    """Write content to a file."""
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
    return f"Written to '{path}'."


def execute_python(code: str) -> str:
    """Execute Python code and return output."""
    try:
        result = subprocess.run(
            [sys.executable, "-c", code],
            capture_output=True,
            text=True,
            timeout=30,
        )
        if result.returncode != 0:
            return f"Exit code {result.returncode}\nSTDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}".strip()
        return result.stdout.strip() or "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Code execution timed out (30s limit)."

def show_python(code: str):
    """Print syntax-highlighted Python code."""
    print(highlight(code, PythonLexer(), TerminalFormatter()))
    return "Shown code with syntax highlighting."

Note that the `return` part of every function is essentially the `OBSERVATION` that is being returned to your `TinyAgent`!

Next, we define these tools and add them to your `NativeTools`:

In [5]:
from illustrated_agents.chapters.ch5_native import NativeTools

tools = NativeTools()
tools.add_tool("read_file", read_file)
tools.add_tool("list_files", list_files)
tools.add_tool("write_file", write_file)
tools.add_tool("execute_python", execute_python)
tools.add_tool("show_python", show_python)

## 3 - Too Much Autonomy?

Although we now have a number of tools, some of them can really mess with your enviroment. Writing to files and especially executing python functions is something that shouldn't be done arbritarily. As such, we are going to update `Tools` and `NativeTools` with a simple safeguard. The safeguard will be that certain tools will require approval before they can be run. LLMs are not perfect and might accidentally do things they really shouldn't do, such as updating their own configurations thereby breaking it... true story!

Either way, this update is rather straightforward, the updated `Tools` class is now as follows but note that we only made marginal changes (more and that later!):

In [6]:
from illustrated_agents.chapters.ch2 import Response
from illustrated_agents.chapters import ch6


class Tools(ch6.Tools):
    """Tool registry for the Agent."""

    def __init__(self, requires_approval: list[str] = []):
        """Initialize Tools

        Arguments:
            requires_approval: A list of tool names that require human approval
        """
        self.registry = {}
        self.requires_approval = requires_approval

    def execute(self, response: Response) -> any:
        """Run a registered tool.

        Arguments:
            tool_call: A parsed tool call dict with "tool" and "kwargs" keys.
        """
        tool_call = response.tool_call
        name, kwargs = tool_call["tool"], tool_call.get("kwargs", {})

        # Human-in-the-loop: ask before running dangerous tools
        if name in self.registry and name in self.requires_approval:
            response = input(f"Allow {name}? [y/N] ").strip().lower()
            if response not in ("y", "yes"):
                return f"Tool '{name}' was denied by the user."

        # Handle registered tools
        if name in self.registry:
            tool_func = self.registry[name]["function"]
            return tool_func(**kwargs)


The only change we made is to `run` to check whether a tool requires approval. If it does, then the user will need to reply with either `"y"` or `"yes"`:

In [7]:
from illustrated_agents.chapters.ch11 import tool_diff; tool_diff

Let's explore it a bit closer:

In [8]:
from illustrated_agents.chapters.ch11 import tools_annotated; tools_annotated

Now we can redefine the `NativeTools` by having it inherit this behavior again from `Tools` which now uses `requires_approval` to check which functions require specific approval from the user before they can be run.

In [9]:
import json
from illustrated_agents.chapters.ch5_native import tool_to_schema

class NativeTools(Tools):
    """Tool registry using native function calling."""

    @property
    def schemas(self) -> list:
        """Return tool functions for native function calling."""
        return [tool_to_schema(tool["function"]) for tool in self.registry.values()]

    @property
    def prompt(self) -> str:
        """Empty because we don't need a prompt for native tool calling"""
        return ""

    def parse(self, response: Response) -> Response:
        """Parse a tool call."""
        # If there's no tool call, return the response as is
        if not response.tool_call:
            return response

        # Extract the tool name and arguments from the tool call
        args = response.tool_call["function"]["arguments"]
        if isinstance(args, str):
            args = json.loads(args)
        tool_call = {"tool": response.tool_call["function"]["name"], "kwargs": args}

        # Add the parsed tool call to the response
        return Response(
            content=response.content,
            reasoning=response.reasoning,
            tool_call=tool_call,
        )

    def observation(self, result):
        """Native tool results use the 'tool' role."""
        return "tool", str(result)

    def is_done(self, response: Response) -> bool:
        """No tool call means the `TinyAgent` is done."""
        return not response.tool_call

Let's redefine our `NativeTools` and this time make sure that the `write` and `execute` tools require approval:

In [10]:
tools = NativeTools(requires_approval=["write_file", "execute_python"])
tools.add_tool("read_file", read_file)
tools.add_tool("list_files", list_files)
tools.add_tool("write_file", write_file)
tools.add_tool("execute_python", execute_python)
tools.add_tool("show_python", show_python)

Note that the write and execute tools are marked as `requires_approval`. As discussed before, this means that everytime your `TinyAgent` wants to run that particular, it will ask confirmation from you before continuing. 

Let's try it out and see if it would try to execute some python code:

In [11]:
# Define tool calll
response = Response(
    content="Some content...",
    reasoning="Some reasoning...",
    tool_call= {
        "tool": "execute_python",
        "kwargs": {"code": "print('Hello World!')"}
    }
)

# Execute Tool
tools.execute(response)

Allow execute_python? [y/N]  


"Tool 'execute_python' was denied by the user."

## 4 - The Styling

To create a coding agent, you will need a nicely looking interface for us to use so you can easily see the models thoughts, actions, and observations. For that, we make use of the [`rich`](https://github.com/Textualize/rich) package library for rich text and beautiful formatting in the terminal.

We create a `Display` class that animates variious parts of the thinking/answering process. This `Display` is merely used for styling:

In [12]:
from rich.console import Console
from rich.rule import Rule

from illustrated_agents.chapters.ch2 import Response

console = Console()

class Display:
    """Handles agent events with Rich formatting."""

    def __call__(self, event: str, data: str | Response = None) -> None:
        # Animate "thinking"
        if event == "thinking":
            console.print("  [dim]Thinking...[/]")

        # Print THOUGHT
        elif event == "response":
            console.print(f"  [bold dark_orange]{'THOUGHT':<13}[/][dim italic]{data.reasoning}[/]")

            if data.content:
                console.print(f"  [bold cyan]{'ANSWER':<13}[/][dim italic]{data.content}[/]")

        # Print ACTION
        elif event == "tool_call" and data.tool_call:
            tool = data.tool_call["tool"]
            kwargs = data.tool_call["kwargs"]
            console.print(f"  [bold yellow]{'ACTION':<13}[/][yellow]{tool}({str(kwargs)})[/]")

        # Print OBSERVATION
        elif event == "observation":
            console.print(f"  [bold green]{'OBSERVATION':<13}[/]{data}")
            console.print(Rule(style="dim"), end="\n\n")

Let's take a closer look at what it is doing and why:

In [13]:
from illustrated_agents.chapters.ch11 import display_annotated; display_annotated

To further illustrate this `Display`, we can mimic the behavior of the Agent and have it "generate" a `THOUGHT`, `ACTION`, `TOOL_CALL`, and `OBSERVATION`.

In [14]:
# Let's create a Response with a tool call and no final answer
response = Response(
    content="",
    reasoning="Let's execute some python!",
    tool_call= {
        "tool": "execute_python",
        "kwargs": {"code": "print('Hello World!')"}
    }
)


# Display the Response
display = Display()
display("thinking", response)
display("response", response)
display("tool_call", response)
display("observation", "Hello World!")

Thinking...

THOUGHT      Let's execute some python!

ACTION       execute_python({'code': "print('Hello World!')"})

OBSERVATION  Hello World!

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

We can also show what the final answer would look like:

In [15]:
# Let's create a Response with a final answer and no tool call
response = Response(
    content="I believe that one plus one is two!",
    reasoning="I should answer the question.",
)

# Display the Response
display = Display()
display("thinking", response)
display("response", response)

Thinking...

THOUGHT      I should answer the question.

ANSWER       I believe that one plus one is two!

## 5 - The `TinyAgent`

The `TinyAgent` will also need an update so that it can use the `Display` whenever it is finished creating `THOUGHT`, `ACTION`, `TOOL_CALL`, and `OBSERVATION`.

The `TinyAgent` only requires adding `self.display(...)` at various places to track what is happening

Since the changes to the `TinyAgent` are minimal and requires adding the `Display` and calling it in specific places whenever the `TinyAgent` has completed something.

In [16]:
from illustrated_agents.chapters.ch11 import tinyagents_diff; tinyagents_diff

Which makes the `TinyAgent` the following updated class:

In [17]:
from illustrated_agents.chapters.ch2 import Response
from illustrated_agents.chapters.ch5_native import LLM, tool_to_schema
from illustrated_agents.chapters.ch6_skills import Skills
from illustrated_agents.chapters.ch6 import ReAct
from illustrated_agents.chapters.ch9 import Memory

class TinyAgent:
    """A minimal, modular, and educational agent framework."""

    def __init__(
        self, llm: LLM, memory: Memory, tools: Tools, planner: ReAct, skills: Skills, display=Display
    ):
        self.llm = llm
        self.memory = memory
        self.tools = tools
        self.planner = planner
        self.skills = skills
        self.display = display

        # Build system prompt with all components
        system_prompt = "You are a helpful assistant.\n"
        system_prompt += self.planner.prompt + "\n"
        system_prompt += self.tools.prompt + "\n"
        system_prompt += self.skills.prompt + "\n"
        self.memory.add("system", system_prompt)

    def run(self, task: str, image_data: str = None) -> str:
        """Run the agent on a task."""
        self.memory.add("user", task, image_data=image_data)

        # `Autonomy` loop
        for step in range(self.planner.max_steps):
            result = self._step()
            if result is not None:
                return result

        return "Max steps reached without completion."

    def _step(self) -> str:
        """Perform a single step."""
        # THOUGHT: Generate response and add to memory
        response = self.llm.generate(self.memory.get_messages(), tools=self.tools.schemas)
        self.memory.add("assistant", response.content, tool_call=response.tool_call)
        self.display("response", response)

        # Tool parsing
        response = self.tools.parse(response)

        # ANSWER: Stopping mechanism
        if self.tools.is_done(response):
            return response.content

        return self._execute_action(response)

    def _execute_action(self, response: Response) -> None:
        """Execute a tool action."""

        # ACTION: execute tools
        self.display("tool_call", response)
        result = self.tools.execute(response)

        # OBSERVATION: add tool results to memory and display
        role, observation = self.tools.observation(result)
        self.memory.add(role, observation)
        self.display("observation", observation)

        return None

## 6 - Putting It All Together

In [18]:
from illustrated_agents.chapters.ch6_native import NativeReAct
from illustrated_agents.chapters.ch6_skills import Skills

# Multimodal Agent
agent = TinyAgent(
    llm=llm, 
    tools=tools, 
    memory=Memory(),
    skills=Skills(),
    planner=NativeReAct(),
    display=display
)

In [19]:
# Run your Coding Agent
## NOTE: You will be asked to allow the agent to execute Python code. 
## Check the output and type `y` if you want to allow it.
_ = agent.run("Create a simple function that uses `Rich` to show formatted Python. Show the code to me before writing it to `rich.py`.")

THOUGHT      The user wants a Python function that uses the `rich` library for formatted output when displaying 
Python code.
They also specified two steps:
1.  Show the code to the user first.
2.  Write the code to a file named `rich.py`.

I should use the `show_python` tool to display the code first, as requested.
Then, I will construct the Python code that encapsulates the requested functionality and use the `write_file` tool 
to save it to `rich.py`.

**Plan:**
1.  Write the Python code for the function.
2.  Call `show_python` with the code.
3.  Call `write_file` with the code and the path `rich.py`.

**Code drafting:**

The function should take a string of code and use `rich` to display it nicely. `rich.print()` combined with code 
formatting or directly using `rich.syntax` (or a similar mechanism if available/appropriate for a simple example) 
would work. A basic approach is to use `rich.print(code_string)` but making it look like code is better. Since the 
request is to "show formatted Python," I will use `rich.print` with a dedicated function call that uses 
`rich.Syntax` or just structure the print statement to imply code formatting.

Let's use `rich.print(f"```python\n{code_to_show}\n```")` or, even better, use `rich.syntax` if I can assume its 
usage, but for simplicity, I will focus on creating a runnable example that demonstrates *using* `rich` for 
formatted output.

A simple function structure:
```python
from rich.console import Console
from rich.syntax import Syntax

def display_python_code_formatted(code_snippet: str):
    """Displays a string of code using rich for syntax highlighting."""
    console = Console()
    syntax = Syntax(code_snippet, "python", theme="monokai", line_numbers=True)
    console.print(syntax)

if __name__ == "__main__":
    example_code = """
def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        print(a)
        a, b = b, a + b
"""
    print("--- Displaying Example Code ---")
    display_python_code_formatted(example_code)
```

I will use this code for both steps.

ACTION       show_python({'code': 'from rich.console import Console\nfrom rich.syntax import Syntax\n\ndef 
display_python_code_formatted(code_snippet: str):\n    """Displays a string of code using rich for syntax 
highlighting."""\n    console = Console()\n    # Use Syntax object for proper formatting\n    syntax = 
Syntax(code_snippet, "python", theme="monokai", line_numbers=True)\n    console.print(syntax)\n\nif __name__ == 
"__main__":\n    example_code = """\ndef fibonacci(n):\n    a, b = 0, 1\n    for _ in range(n):\n        print(a)\n
a, b = b, a + b\n"""\n    print("--- Displaying Example Code ---")\n    
display_python_code_formatted(example_code)'})

from rich.console import Console
from rich.syntax import Syntax

def display_python_code_formatted(code_snippet: str):
    """Displays a string of code using rich for syntax highlighting."""
    console = Console()
    # Use Syntax object for proper formatting
    syntax = Syntax(code_snippet, "python", theme="monokai", line_numbers=True)
    console.print(syntax)

if __name__ == "__main__":
    example_code = """
def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        print(a)
        a, b = b, a + b
"""
    print("--- Displaying Example Code ---")
    display_python_code_formatted(example_code)



OBSERVATION  Shown code with syntax highlighting.

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

Allow write_file? [y/N]  n


OBSERVATION  Tool 'write_file' was denied by the user.

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

THOUGHT      None

ANSWER       I have shown you the Python code using `Rich` for formatted output using the `show_python` tool, and
I attempted to write it to `rich.py` using the `write_file` tool.

Since the `write_file` tool was denied, here is the content of the function again:

### `rich.py`

```python
from rich.console import Console
from rich.syntax import Syntax

def display_python_code_formatted(code_snippet: str):
    """Displays a string of code using rich for syntax highlighting."""
    console = Console()
    # Use Syntax object for proper formatting
    syntax = Syntax(code_snippet, "python", theme="monokai", line_numbers=True)
    console.print(syntax)

if __name__ == "__main__":
    example_code = """
def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        print(a)
        a, b = b, a + b
"""
    print("--- Displaying Example Code ---")
    display_python_code_formatted(example_code)
```

This function, when executed, will require the `rich` library to be installed (`pip install rich`) to function 
correctly. It defines `display_python_code_formatted` to take any code string and print it to the console with 
proper syntax highlighting.

And there you have it! Your own Coding Agent that can use various tools, run python code, and even allows you to judge whether you want the code to be executed. As always, let's explore the messages:

In [20]:
from rich import print as pprint

pprint(agent.memory.messages)

[
    {'role': 'system', 'content': 'You are a helpful assistant.\n\n\n\n'},
    {
        'role': 'user',
        'content': 'Create a simple function that uses `Rich` to show formatted Python. Show the code to me before 
writing it to `rich.py`.'
    },
    {
        'role': 'assistant',
        'content': '',
        'tool_calls': [
            {
                'id': 'call_xs97igly',
                'function': {
                    'arguments': '{"code":"from rich.console import Console\\nfrom rich.syntax import 
Syntax\\n\\ndef display_python_code_formatted(code_snippet: str):\\n    \\"\\"\\"Displays a string of code using 
rich for syntax highlighting.\\"\\"\\"\\n    console = Console()\\n    # Use Syntax object for proper formatting\\n
syntax = Syntax(code_snippet, \\"python\\", theme=\\"monokai\\", line_numbers=True)\\n    
console.print(syntax)\\n\\nif __name__ == \\"__main__\\":\\n    example_code = \\"\\"\\"\\ndef fibonacci(n):\\n    
a, b = 0, 1\\n    for _ in range(n):\\n        print(a)\\n        a, b = b, a + b\\n\\"\\"\\"\\n    print(\\"--- 
Displaying Example Code ---\\")\\n    display_python_code_formatted(example_code)"}',
                    'name': 'show_python'
                },
                'type': 'function',
                'index': 0
            }
        ]
    },
    {'role': 'tool', 'content': 'Shown code with syntax highlighting.'},
    {
        'role': 'assistant',
        'content': '',
        'tool_calls': [
            {
                'id': 'call_m6x6e28v',
                'function': {
                    'arguments': '{"content":"from rich.console import Console\\nfrom rich.syntax import 
Syntax\\n\\ndef display_python_code_formatted(code_snippet: str):\\n    \\\\\\"\\\\\\"\\\\\\"Displays a string of 
code using rich for syntax highlighting.\\\\\\"\\\\\\"\\\\\\"\\n    console = Console()\\n    # Use Syntax object 
for proper formatting\\n    syntax = Syntax(code_snippet, \\"python\\", theme=\\"monokai\\", line_numbers=True)\\n 
console.print(syntax)\\n\\nif __name__ == \\"__main__\\":\\n    example_code = \\\\\\"\\\\\\"\\\\\\"\\ndef 
fibonacci(n):\\n    a, b = 0, 1\\n    for _ in range(n):\\\\n        print(a)\\\\n        a, b = b, a + 
b\\n\\\\\\"\\\\\\"\\\\\\"\\n    print(\\"--- Displaying Example Code ---\\")\\n    
display_python_code_formatted(example_code)","path":"rich.py"}',
                    'name': 'write_file'
                },
                'type': 'function',
                'index': 0
            }
        ]
    },
    {'role': 'tool', 'content': "Tool 'write_file' was denied by the user."},
    {
        'role': 'assistant',
        'content': 'I have shown you the Python code using `Rich` for formatted output using the `show_python` 
tool, and I attempted to write it to `rich.py` using the `write_file` tool.\n\nSince the `write_file` tool was 
denied, here is the content of the function again:\n\n### `rich.py`\n\n```python\nfrom rich.console import 
Console\nfrom rich.syntax import Syntax\n\ndef display_python_code_formatted(code_snippet: str):\n    """Displays a
string of code using rich for syntax highlighting."""\n    console = Console()\n    # Use Syntax object for proper 
formatting\n    syntax = Syntax(code_snippet, "python", theme="monokai", line_numbers=True)\n    
console.print(syntax)\n\nif __name__ == "__main__":\n    example_code = """\ndef fibonacci(n):\n    a, b = 0, 1\n  
for _ in range(n):\n        print(a)\n        a, b = b, a + b\n"""\n    print("--- Displaying Example Code ---")\n 
display_python_code_formatted(example_code)\n```\n\nThis function, when executed, will require the `rich` library 
to be installed (`pip install rich`) to function correctly. It defines `display_python_code_formatted` to take any 
code string and print it to the console with proper syntax highlighting.'
    }
]

# What We Built

In this chapter, we covered a major step into creating a helpful assistant, namely by adding coding capabilities! We did that through XML-based tools and made sure that this could be done safely with a human-in-the-loop. 

In [1]:
from illustrated_agents.chapters.ch11 import what_we_built; what_we_built

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py      ← Updated (Added printing to the `Display` to see its intermediate steps.)                    │
│ ├── display.py    ← New (A new `Display` class to format the agent's THOUGHTS, ACTIONS, and OBSERVATIONS.)      │
│ ├── llm.py                                                                                                      │
│ ├── memory.py                                                                                                   │
│ ├── planning.py                                                                                                 │
│ ├── reflection.py                                                                                               │
│ ├── skills.py                                                                                                   │
│ ├── toolbox.py    ← Updated (Added tools for Coding Agents.)                                                    │
│ └── tools.py      ← Updated (Added XML-based tool parsing and calling with human-in-the-loop checks.)           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯